# 04 — Extreme Value Theory: Tail Risk with the Generalised Pareto Distribution

**Extreme Value Theory (EVT)** is the branch of statistics concerned with modelling the *tails* of distributions — the rare, extreme events that standard models systematically underestimate.

For risk managers, EVT matters because:

- **Normal-based (Parametric) VaR** assumes thin tails and dramatically underestimates losses at 99%+ confidence.
- **Historical VaR** is limited by the sample — if your worst observed loss is -5%, it cannot estimate a -7% loss.
- **EVT** extrapolates into the tail using the Generalised Pareto Distribution (GPD), providing estimates even for events *worse than anything observed*.

The **Peaks-over-Threshold (POT)** approach fits a GPD to losses exceeding a high threshold $u$, characterised by:

$$
G_{\xi,\beta}(x) = 1 - \left(1 + \xi \frac{x}{\beta}\right)^{-1/\xi}
$$

where $\xi$ is the **shape parameter** (tail heaviness) and $\beta$ is the **scale parameter**.

---

In [ ]:
import sys
sys.path.insert(0, "../src")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

from var_risk_engine.evt import (
    fit_gpd,
    evt_var,
    evt_es,
    compare_evt_methods,
    gpd_qq_plot,
)
from var_risk_engine.var_historical import historical_var
from var_risk_engine.var_parametric import parametric_var
from var_risk_engine.expected_shortfall import expected_shortfall, es_parametric

%matplotlib inline

# Color palette
PRIMARY    = "#1B3A5C"
SECONDARY  = "#E8734A"
TERTIARY   = "#4CAF50"
QUATERNARY = "#9C27B0"

sns.set_theme(style="whitegrid", font_scale=1.1)
plt.rcParams["figure.figsize"] = (12, 6)
plt.rcParams["figure.dpi"] = 120

In [ ]:
# --- Generate synthetic fat-tailed returns ---
# Student-t with df=5 has excess kurtosis = 6/(df-4) = 6, modelling
# the heavy tails commonly observed in equity returns.
np.random.seed(42)
n_obs = 2000

# Simulate daily returns: location=-0.0002 (slight negative drift),
# scale=0.015 (~1.5% daily vol), df=5 (fat tails).
raw = stats.t.rvs(df=5, loc=-0.0002, scale=0.015, size=n_obs)
returns = np.asarray(raw, dtype=float)
losses = -returns

print(f"Synthetic return sample: {n_obs} observations")
print(f"Mean return:   {returns.mean():.6f}")
print(f"Std return:    {returns.std():.6f}")
print(f"Skewness:      {stats.skew(returns):.4f}")
print(f"Kurtosis:      {stats.kurtosis(returns):.4f}  (normal = 0)")
print(f"Min return:    {returns.min():.6f}")
print(f"Max loss:      {losses.max():.6f}")

# Visualise the return distribution
fig, ax = plt.subplots(figsize=(12, 5))
ax.hist(returns, bins=80, density=True, alpha=0.55,
        color=PRIMARY, edgecolor="white", label="Simulated returns")

xmin, xmax = ax.get_xlim()
x_kde = np.linspace(xmin, xmax, 500)
kde = stats.gaussian_kde(returns)
ax.plot(x_kde, kde(x_kde), color="black", linewidth=1.8, label="KDE")

# Overlay fitted normal for comparison
mu, sigma = returns.mean(), returns.std()
ax.plot(x_kde, stats.norm.pdf(x_kde, mu, sigma),
        color=SECONDARY, linewidth=1.8, linestyle="--", label="Fitted Normal")

ax.set_xlabel("Daily Return")
ax.set_ylabel("Density")
ax.set_title("Synthetic Fat-Tailed Returns (Student-t, df=5) vs Normal Fit")
ax.legend(loc="upper left", frameon=True, framealpha=0.9)
plt.tight_layout()
plt.show()

In [ ]:
# --- Fit GPD to tail losses ---
gpd_params = fit_gpd(losses)  # auto-threshold at 90th percentile

print("=" * 60)
print("  GPD Fit Results (Peaks-over-Threshold)")
print("=" * 60)
print(f"  Shape (xi):          {gpd_params['xi']:.4f}")
print(f"  Scale (beta):        {gpd_params['beta']:.4f}")
print(f"  Threshold (u):       {gpd_params['threshold']:.4f}")
print(f"  Exceedances:         {gpd_params['n_exceedances']} / {gpd_params['n_total']}")
print(f"  Exceedance rate:     {gpd_params['exceedance_rate']:.4f}")
print(f"  KS statistic:        {gpd_params['ks_statistic']:.4f}")
print(f"  KS p-value:          {gpd_params['ks_pvalue']:.4f}")
print("=" * 60)

if gpd_params['ks_pvalue'] > 0.05:
    print("\n  KS test: fail to reject H0 at 5% \u2014 GPD is a reasonable fit.")
else:
    print("\n  KS test: reject H0 at 5% \u2014 GPD fit may be inadequate.")

if gpd_params['xi'] > 0:
    print(f"\n  xi > 0 indicates a heavy (Fr\u00e9chet-type) tail.")
    print(f"  Tail index alpha = 1/xi = {1/gpd_params['xi']:.2f}")
elif gpd_params['xi'] < 0:
    print(f"\n  xi < 0 indicates a bounded (Weibull-type) tail.")
else:
    print(f"\n  xi = 0 indicates an exponential (Gumbel-type) tail.")

In [ ]:
# --- Compute EVT-VaR and EVT-ES at multiple confidence levels ---
confidence_levels = [0.95, 0.975, 0.99, 0.995, 0.999]

rows = []
for cl in confidence_levels:
    v_hist = historical_var(returns, confidence=cl)
    es_hist = expected_shortfall(returns, confidence=cl)
    v_para = parametric_var(returns, confidence=cl)
    es_para = es_parametric(returns, confidence=cl)
    v_evt = evt_var(gpd_params, confidence=cl)
    es_evt = evt_es(gpd_params, confidence=cl)

    rows.append({
        "confidence": cl,
        "VaR_Historical": v_hist,
        "ES_Historical": es_hist,
        "VaR_Parametric": v_para,
        "ES_Parametric": es_para,
        "VaR_EVT": v_evt,
        "ES_EVT": es_evt,
    })

evt_table = pd.DataFrame(rows)

print("\n  VaR & ES Comparison: Historical vs Parametric vs EVT")
print("  " + "=" * 90)
print(evt_table.to_string(index=False, float_format="%.5f"))
print("  " + "=" * 90)

# Show the EVT premium at the highest confidence level
last = evt_table.iloc[-1]
evt_premium = last["VaR_EVT"] - last["VaR_Parametric"]
print(f"\n  EVT premium over Parametric VaR at {last['confidence']:.1%}: "
      f"{evt_premium:.5f} ({evt_premium/last['VaR_Parametric']*100:.1f}% higher)")

In [ ]:
# --- One-call comparison using compare_evt_methods ---
comparison_df = compare_evt_methods(returns)

print("\n  compare_evt_methods() output:")
print("  " + "-" * 90)
print(comparison_df.to_string(index=False, float_format="%.5f"))
print("  " + "-" * 90)

# Highlight: at 99.9% confidence, how much higher is EVT-VaR vs Historical?
row_999 = comparison_df[comparison_df["confidence"] == 0.999].iloc[0]
ratio = row_999["var_evt"] / row_999["var_historical"]
print(f"\n  At 99.9% confidence:")
print(f"    EVT-VaR / Historical VaR = {ratio:.2f}x")
print(f"    EVT extrapolates beyond observed extremes using the GPD tail model.")

In [ ]:
# --- GPD Q-Q plot: empirical vs theoretical quantiles ---
fig, ax = gpd_qq_plot(losses, gpd_params)
plt.show()

print("A good GPD fit shows points close to the 45-degree line.")
print("Deviations in the upper-right corner indicate the model")
print("under- or over-estimates the most extreme exceedances.")

In [ ]:
# --- Bar chart: VaR methods across confidence levels ---
cl_labels = [f"{c:.1%}" for c in comparison_df["confidence"]]
x = np.arange(len(cl_labels))
width = 0.25

fig, ax = plt.subplots(figsize=(13, 6))

bars1 = ax.bar(x - width, comparison_df["var_historical"], width,
               label="Historical VaR", color=PRIMARY, edgecolor="white")
bars2 = ax.bar(x, comparison_df["var_parametric"], width,
               label="Parametric VaR", color=SECONDARY, edgecolor="white")
bars3 = ax.bar(x + width, comparison_df["var_evt"], width,
               label="EVT-VaR", color=TERTIARY, edgecolor="white")

ax.set_xlabel("Confidence Level", fontsize=12)
ax.set_ylabel("VaR (positive = loss)", fontsize=12)
ax.set_title("VaR Comparison: Historical vs Parametric vs EVT", fontsize=14)
ax.set_xticks(x)
ax.set_xticklabels(cl_labels)
ax.legend(loc="upper left", frameon=True, framealpha=0.9)

# Annotate the EVT premium at the highest confidence level
max_idx = len(cl_labels) - 1
evt_val = comparison_df["var_evt"].iloc[max_idx]
para_val = comparison_df["var_parametric"].iloc[max_idx]
ax.annotate(
    f"EVT premium\n+{evt_val - para_val:.4f}",
    xy=(max_idx + width, evt_val),
    xytext=(max_idx - 0.5, evt_val + 0.005),
    arrowprops=dict(arrowstyle="->", color="gray"),
    fontsize=10, color=TERTIARY, fontweight="bold",
)

plt.tight_layout()
plt.show()

print("Notice how the gap between methods widens at higher confidence levels.")
print("This is exactly where EVT's tail extrapolation adds the most value.")

---

## Conclusion: Why EVT Matters at High Confidence

The key takeaway from this analysis is that **the choice of VaR method matters most at extreme confidence levels** (99% and above):

1. **Parametric VaR** assumes a normal distribution and systematically underestimates tail risk. At 99.9% confidence, the normal distribution expects losses beyond 3.09 standard deviations \u2014 but fat-tailed equity returns produce such events far more frequently.

2. **Historical VaR** is limited by the sample size. With 2,000 observations, the 99.9% VaR is essentially the single worst loss observed \u2014 a very noisy estimate that cannot extrapolate beyond the data.

3. **EVT-VaR** fits a parametric model (GPD) specifically to the tail, then uses it to *extrapolate* beyond observed extremes. The GPD shape parameter $\xi$ directly controls tail heaviness: when $\xi > 0$, the tail decays polynomially rather than exponentially, producing much larger VaR estimates at high confidence.

**The EVT premium** (EVT-VaR minus Parametric VaR) grows with confidence because the normal distribution's tail decays as $e^{-x^2/2}$ while the GPD with $\xi > 0$ decays as $x^{-1/\xi}$ \u2014 a much slower rate. For risk managers concerned with worst-case scenarios, EVT provides a more prudent and theoretically grounded estimate of extreme losses.

**Practical considerations:**
- EVT requires enough tail observations (typically 50+ exceedances) for stable GPD fitting.
- The threshold selection involves a bias-variance tradeoff: too low and the GPD approximation breaks down; too high and there are too few exceedances.
- EVT should complement, not replace, Historical and Parametric methods \u2014 triangulating across methods gives the most robust risk picture.